In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np 
import os                                              
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

In [2]:
df=pd.read_csv("../Data/raw/Data_Entry_2017_v2020.csv")

In [3]:
df = df.rename(columns={
    "Image Index": "image_name",
    "Finding Labels": "finding_labels",
    "Follow-up #": "follow_up",
    "Patient ID": "patient_id",
    "Patient Age": "patient_age",
    "Patient Sex": "patient_sex",
    "View Position": "view_position",
    "OriginalImage[Width": "image_width",
    "Height]": "image_height",
    "OriginalImagePixelSpacing[x": "pixel_spacing_x",
    "y]": "pixel_spacing_y",
})


In [4]:
df["label"] = (
    df["finding_labels"].str.strip().ne("No Finding")
).astype(int)

In [5]:

# =====================================================
# CONFIGURATION
# =====================================================
SEED = 42
TOTAL_IMAGES = 20_000

SPLIT_TARGETS = {
    "train":      {"Normal": 7000, "Abnormal": 7000},
    "validation": {"Normal": 1500, "Abnormal": 1500},
    "test":       {"Normal": 1500, "Abnormal": 1500},
}

# Disease balancing strength:
# 1.00 = preserve natural disease proportions more closely
# 0.75 = moderately prioritize less frequent diseases
# Lower values = stronger balancing pressure
BALANCE_ALPHA = 0.75


# =====================================================
# 1. PREPARE METADATA AND LABELS
# =====================================================
data = df.copy().reset_index(drop=True)

required_cols = [
    "image_name",
    "finding_labels",
    "patient_id"
]

missing_cols = [
    col for col in required_cols
    if col not in data.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )

if data["patient_id"].isna().any():
    raise ValueError(
        "Missing patient_id values found. "
        "Resolve them before splitting."
    )

if data["image_name"].isna().any():
    raise ValueError(
        "Missing image_name values found. "
        "Resolve them before splitting."
    )

if data["image_name"].duplicated().any():
    raise ValueError(
        "Duplicate image_name values found. "
        "Resolve duplicates first."
    )


def parse_findings(value):
    """Convert a pipe-separated finding string into a set."""

    if pd.isna(value):
        return set()

    return {
        label.strip()
        for label in str(value).split("|")
        if label.strip()
        and label.strip().lower() != "nan"
    }


data["_findings"] = data["finding_labels"].apply(
    parse_findings
)

# Stop if any image has missing or unusable labels.
empty_labels = data["_findings"].apply(len).eq(0)

if empty_labels.any():
    raise ValueError(
        f"{empty_labels.sum()} rows have empty finding_labels. "
        "Inspect them before proceeding."
    )

# Normal means No Finding is the ONLY label.
# Any disease label means Abnormal.
data["binary_label"] = data["_findings"].apply(
    lambda labels: (
        "Normal"
        if labels == {"No Finding"}
        else "Abnormal"
    )
)

# Extract the 14 disease labels.
diseases = sorted({
    label
    for labels in data["_findings"]
    for label in labels
    if label != "No Finding"
})

disease_cols = [
    f"disease__{disease}"
    for disease in diseases
]

# Create one binary column per disease.
# An image can have multiple disease labels.
for disease, col in zip(diseases, disease_cols):
    data[col] = data["_findings"].apply(
        lambda labels: int(disease in labels)
    )

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print("Images available:", f"{len(data):,}")
print(
    "Unique patients:",
    f"{data['patient_id'].nunique():,}"
)

print("Disease labels:", diseases)

print("\nOriginal binary-label distribution:")
print(data["binary_label"].value_counts())


# =====================================================
# 2. SPLIT PATIENTS FIRST: 70% / 15% / 15%
# =====================================================
# IMPORTANT:
# All images belonging to a patient must stay in one split.

patient_diseases = data.groupby("patient_id")[
    disease_cols
].max()

patient_binary = pd.crosstab(
    data["patient_id"],
    data["binary_label"]
).reindex(
    columns=["Normal", "Abnormal"],
    fill_value=0
)

# Patient-level indicators for Normal and Abnormal.
patient_binary = (
    patient_binary > 0
).astype(int)

patient_features = pd.concat(
    [patient_binary, patient_diseases],
    axis=1
)

patient_ids = patient_features.index.to_numpy()

Y = patient_features.to_numpy(dtype=int)

# Split 1: 70% train, 30% temporary.
splitter_1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=SEED
)

train_idx, temp_idx = next(
    splitter_1.split(
        np.zeros((len(patient_ids), 1)),
        Y
    )
)

train_patients = patient_ids[train_idx]
temp_patients = patient_ids[temp_idx]

Y_temp = Y[temp_idx]

# Split 2: divide the temporary patients into validation/test.
splitter_2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=SEED
)

val_idx, test_idx = next(
    splitter_2.split(
        np.zeros((len(temp_patients), 1)),
        Y_temp
    )
)

validation_patients = temp_patients[val_idx]
test_patients = temp_patients[test_idx]

patient_to_split = {}

for patient in train_patients:
    patient_to_split[patient] = "train"

for patient in validation_patients:
    patient_to_split[patient] = "validation"

for patient in test_patients:
    patient_to_split[patient] = "test"

data["split"] = data["patient_id"].map(
    patient_to_split
)

if data["split"].isna().any():
    raise ValueError(
        "Some patients were not assigned to a split."
    )

# Verify patient separation before image selection.
patient_sets = {
    split: set(
        data.loc[
            data["split"] == split,
            "patient_id"
        ]
    )
    for split in ["train", "validation", "test"]
}

assert patient_sets["train"].isdisjoint(
    patient_sets["validation"]
)

assert patient_sets["train"].isdisjoint(
    patient_sets["test"]
)

assert patient_sets["validation"].isdisjoint(
    patient_sets["test"]
)

print("\n" + "=" * 60)
print("PATIENT SPLIT SIZES")
print("=" * 60)

for split, patients in patient_sets.items():
    print(
        f"{split:12s}: {len(patients):,} patients"
    )


# =====================================================
# 3. DISEASE-AWARE IMAGE SELECTION
# =====================================================
def select_disease_aware_sample(
    pool,
    target,
    label_columns,
    seed,
    alpha=0.75
):
    """
    Select exactly 'target' unique images.

    The algorithm prioritizes diseases with the largest
    relative shortfall against their desired counts.

    Images retain ALL their original disease labels.
    A single image can contribute to multiple diseases.

    alpha:
        1.00 -> targets closer to natural disease prevalence
        0.75 -> moderately prioritizes rare diseases
        Lower values -> stronger balancing pressure

    Note:
        This is a heuristic. It does not guarantee equal
        disease counts or an optimal global solution.
    """

    pool = pool.reset_index(drop=True).copy()

    if target < 0:
        raise ValueError(
            "target must be non-negative."
        )

    if target > len(pool):
        raise ValueError(
            f"Need {target:,} images, but only "
            f"{len(pool):,} are available."
        )

    if not 0 <= alpha <= 1:
        raise ValueError(
            "alpha must be between 0 and 1."
        )

    if target == 0:
        return pool.iloc[0:0].copy()

    if target == len(pool):
        return pool.copy()

    # Image x disease matrix.
    Y_pool = pool[label_columns].to_numpy(
        dtype=np.int8
    )

    # Number of available images for each disease.
    available_counts = Y_pool.sum(axis=0)

    # Disease prevalence within this split's abnormal pool.
    prevalence = (
        available_counts / len(pool)
    )

    # Desired counts for each disease.
    # Lower prevalence is boosted when alpha < 1.
    desired_counts = np.ceil(
        target * np.power(prevalence, alpha)
    ).astype(int)

    # Never target more images than are available.
    desired_counts = np.minimum(
        desired_counts,
        available_counts
    )

    rng = np.random.default_rng(seed)

    # Randomize candidates for each disease.
    candidate_lists = [
        rng.permutation(
            np.flatnonzero(Y_pool[:, j])
        )
        for j in range(Y_pool.shape[1])
    ]

    pointers = np.zeros(
        Y_pool.shape[1],
        dtype=int
    )

    selected_mask = np.zeros(
        len(pool),
        dtype=bool
    )

    selected_indices = []

    # Counts of all disease labels represented so far.
    current_counts = np.zeros(
        Y_pool.shape[1],
        dtype=int
    )

    while len(selected_indices) < target:

        # Remaining target for each disease.
        deficits = np.maximum(
            desired_counts - current_counts,
            0
        )

        active_diseases = np.flatnonzero(
            deficits > 0
        )

        # If all targets are met, fill remaining slots
        # randomly without replacement.
        if len(active_diseases) == 0:

            remaining = np.flatnonzero(
                ~selected_mask
            )

            needed = (
                target - len(selected_indices)
            )

            if needed > len(remaining):
                raise RuntimeError(
                    "Not enough unselected images remain."
                )

            extra = rng.choice(
                remaining,
                size=needed,
                replace=False
            )

            selected_indices.extend(
                extra.tolist()
            )

            selected_mask[extra] = True

            break

        # Prioritize the disease with the largest
        # relative shortfall against its target.
        relative_deficits = (
            deficits[active_diseases]
            / np.maximum(
                desired_counts[active_diseases],
                1
            )
        )

        disease_idx = active_diseases[
            np.argmax(relative_deficits)
        ]

        candidates = candidate_lists[
            disease_idx
        ]

        # Find an unused image containing this disease.
        while (
            pointers[disease_idx]
            < len(candidates)
        ):

            image_idx = candidates[
                pointers[disease_idx]
            ]

            pointers[disease_idx] += 1

            if not selected_mask[image_idx]:
                break

        else:
            # No unused candidate remains for this disease.
            desired_counts[disease_idx] = (
                current_counts[disease_idx]
            )

            continue

        # Select the image once.
        selected_mask[image_idx] = True

        selected_indices.append(
            image_idx
        )

        # Update ALL disease counts on this image.
        # This accounts for overlapping multilabel cases.
        current_counts += Y_pool[image_idx]

    chosen = pool.iloc[
        selected_indices
    ].copy()

    # Safety checks.
    assert len(chosen) == target

    assert chosen["image_name"].is_unique

    return chosen


# -----------------------------------------------------
# Select the exact requested number of images per split.
# -----------------------------------------------------
selected_parts = []

for split, targets in SPLIT_TARGETS.items():

    split_data = data[
        data["split"] == split
    ]

    normal_pool = split_data[
        split_data["binary_label"] == "Normal"
    ]

    abnormal_pool = split_data[
        split_data["binary_label"] == "Abnormal"
    ]

    n_normal = targets["Normal"]
    n_abnormal = targets["Abnormal"]

    # Validate availability before sampling.
    if len(normal_pool) < n_normal:
        raise ValueError(
            f"{split}: only {len(normal_pool):,} Normal "
            f"images available; {n_normal:,} required. "
            "Do not move patients between splits."
        )

    if len(abnormal_pool) < n_abnormal:
        raise ValueError(
            f"{split}: only {len(abnormal_pool):,} Abnormal "
            f"images available; {n_abnormal:,} required. "
            "Do not move patients between splits."
        )

    # Normal images: random sampling.
    normal_selected = normal_pool.sample(
        n=n_normal,
        random_state=SEED
    )

    # Abnormal images: disease-aware sampling.
    abnormal_selected = select_disease_aware_sample(
        pool=abnormal_pool,
        target=n_abnormal,
        label_columns=disease_cols,
        seed=SEED,
        alpha=BALANCE_ALPHA
    )

    selected_parts.extend([
        normal_selected,
        abnormal_selected
    ])

    print(
        f"\n{split.upper()}: "
        f"Normal={len(normal_selected):,}, "
        f"Abnormal={len(abnormal_selected):,}"
    )


# Combine selected images and shuffle.
balanced_df = (
    pd.concat(
        selected_parts,
        ignore_index=True
    )
    .sample(
        frac=1,
        random_state=SEED
    )
    .reset_index(drop=True)
)


# =====================================================
# 4. FINAL VALIDATION
# =====================================================
assert len(balanced_df) == TOTAL_IMAGES

assert (
    balanced_df["image_name"].nunique()
    == TOTAL_IMAGES
)

# Confirm exact requested class counts.
actual_counts = pd.crosstab(
    balanced_df["split"],
    balanced_df["binary_label"]
).reindex(
    index=["train", "validation", "test"],
    columns=["Normal", "Abnormal"],
    fill_value=0
)

for split, targets in SPLIT_TARGETS.items():

    for label, target in targets.items():

        actual = int(
            actual_counts.loc[split, label]
        )

        assert actual == target, (
            f"{split}/{label}: expected {target}, "
            f"got {actual}"
        )

# Confirm no patient leakage in the selected dataset.
selected_patient_sets = {
    split: set(
        balanced_df.loc[
            balanced_df["split"] == split,
            "patient_id"
        ]
    )
    for split in ["train", "validation", "test"]
}

assert selected_patient_sets[
    "train"
].isdisjoint(
    selected_patient_sets["validation"]
)

assert selected_patient_sets[
    "train"
].isdisjoint(
    selected_patient_sets["test"]
)

assert selected_patient_sets[
    "validation"
].isdisjoint(
    selected_patient_sets["test"]
)

print("\n" + "=" * 60)
print("FINAL IMAGE COUNTS")
print("=" * 60)

print(actual_counts.to_string())

print(
    "\nTotal images:",
    f"{len(balanced_df):,}"
)

print(
    "Total unique patients:",
    f"{balanced_df['patient_id'].nunique():,}"
)

print("\nUnique patients per selected split:")

for split in ["train", "validation", "test"]:

    count = balanced_df.loc[
        balanced_df["split"] == split,
        "patient_id"
    ].nunique()

    print(f"{split:12s}: {count:,}")

# Disease prevalence among selected abnormal images.
abnormal_selected = balanced_df[
    balanced_df["binary_label"] == "Abnormal"
]

print("\n" + "=" * 60)
print("DISEASE REPRESENTATION IN SELECTED ABNORMAL IMAGES")
print("=" * 60)

print(
    f"Balancing alpha: {BALANCE_ALPHA}"
)

print(
    f"Abnormal images: {len(abnormal_selected):,}"
)

for disease, col in zip(
    diseases,
    disease_cols
):

    count = int(
        abnormal_selected[col].sum()
    )

    percentage = (
        100 * count / len(abnormal_selected)
    )

    print(
        f"{disease:25s} "
        f"{count:6,} "
        f"({percentage:6.2f}%)"
    )

# Final patient leakage check.
print("\nPatient leakage check: PASSED")

print(
    "Duplicate image check:",
    "PASSED"
    if balanced_df["image_name"].is_unique
    else "FAILED"
)


# =====================================================
# 5. SAVE OUTPUTS
# =====================================================
os.makedirs(
    "Data/splits",
    exist_ok=True
)

# Exclude internal helper columns from exported CSVs.
export_cols = [
    col
    for col in balanced_df.columns
    if col != "_findings"
    and not col.startswith("disease__")
]

# Save the complete selected dataset.
balanced_df[export_cols].to_csv(
    "Data/splits/balanced_20k_metadata.csv",
    index=False
)

# Save individual split metadata.
for split in [
    "train",
    "validation",
    "test"
]:

    balanced_df.loc[
        balanced_df["split"] == split,
        export_cols
    ].to_csv(
        f"Data/splits/{split}_metadata.csv",
        index=False
    )

# Save patient assignments for reproducibility.
patient_assignment = pd.DataFrame({
    "patient_id": list(
        patient_to_split.keys()
    ),
    "split": list(
        patient_to_split.values()
    )
})

patient_assignment.to_csv(
    "Data/splits/patient_split_assignments.csv",
    index=False
)

print("\n" + "=" * 60)
print("SAVED FILES")
print("=" * 60)

print(
    "Data/splits/balanced_20k_metadata.csv"
)

print(
    "Data/splits/train_metadata.csv"
)

print(
    "Data/splits/validation_metadata.csv"
)

print(
    "Data/splits/test_metadata.csv"
)

print(
    "Data/splits/patient_split_assignments.csv"
)

print("\nAll processing completed successfully.")

DATASET OVERVIEW
Images available: 112,120
Unique patients: 30,805
Disease labels: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']

Original binary-label distribution:
binary_label
Normal      60361
Abnormal    51759
Name: count, dtype: int64

PATIENT SPLIT SIZES
train       : 21,584 patients
validation  : 4,598 patients
test        : 4,623 patients

TRAIN: Normal=7,000, Abnormal=7,000

VALIDATION: Normal=1,500, Abnormal=1,500

TEST: Normal=1,500, Abnormal=1,500

FINAL IMAGE COUNTS
binary_label  Normal  Abnormal
split                         
train           7000      7000
validation      1500      1500
test            1500      1500

Total images: 20,000
Total unique patients: 10,567

Unique patients per selected split:
train       : 7,365
validation  : 1,595
test        : 1,607

DISEASE REPRESENTATION IN SELECTED ABNORMAL IMAGES
Balancing alpha

In [6]:
# Final selected dataset: patient counts per split
summary = (
    balanced_df.groupby("split")
    .agg(
        images=("image_name", "nunique"),
        patients=("patient_id", "nunique")
    )
    .reindex(["train", "validation", "test"])
)

print("Final 20,000-image subset")
print(summary.to_string())

print("\nPatients and images by split and binary class:")

details = (
    balanced_df.groupby(["split", "binary_label"])
    .agg(
        images=("image_name", "nunique"),
        patients=("patient_id", "nunique")
    )
)

print(details.to_string())

# Confirm there is no patient overlap.
patient_sets = {
    split: set(
        balanced_df.loc[
            balanced_df["split"] == split, "patient_id"
        ]
    )
    for split in ["train", "validation", "test"]
}

assert patient_sets["train"].isdisjoint(patient_sets["validation"])
assert patient_sets["train"].isdisjoint(patient_sets["test"])
assert patient_sets["validation"].isdisjoint(patient_sets["test"])

print("\nPatient leakage check: PASSED")
print("Total unique patients:", balanced_df["patient_id"].nunique())


Final 20,000-image subset
            images  patients
split                       
train        14000      7365
validation    3000      1595
test          3000      1607

Patients and images by split and binary class:
                         images  patients
split      binary_label                  
test       Abnormal        1500       760
           Normal          1500      1116
train      Abnormal        7000      3676
           Normal          7000      5046
validation Abnormal        1500       803
           Normal          1500      1090

Patient leakage check: PASSED
Total unique patients: 10567
